## Interactive Real-time Location Dashboard Template

This template demonstrates how to create a simulated interactive dashboard in a Colab environment to visualize real-time location data (longitude and latitude) without permanent storage. It uses `folium` for interactive maps and `ipywidgets` for basic controls.

In [ ]:
# Install necessary libraries
!pip install folium ipywidgets

In [ ]:
import folium
import random
import time
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from traitlets import Bool

# --- Configuration Parameters ---
INITIAL_LOCATION = [34.0522, -118.2437] # Los Angeles, for example
UPDATE_INTERVAL_SECONDS = 5

# --- Data Simulation Function ---
def generate_random_location(current_lat, current_lon, step_size=0.01):
    """Simulates a new location slightly perturbed from the current one."""
    new_lat = current_lat + random.uniform(-step_size, step_size)
    new_lon = current_lon + random.uniform(-step_size, step_size)
    return [new_lat, new_lon]

# --- Map Visualization Function ---
def create_and_display_map(locations, center_location):
    """Creates a folium map with markers for each location."""
    m = folium.Map(location=center_location, zoom_start=12)
    for lat, lon in locations:
        folium.Marker([lat, lon]).add_to(m)

    # Display the map directly in the output
    display(m)

# --- Live Feed Simulation Function ---
def simulate_live_feed(stop_event, output_widget):
    current_lat, current_lon = INITIAL_LOCATION
    history = []

    with output_widget:
        while not stop_event.value:
            clear_output(wait=True)
            current_location = generate_random_location(current_lat, current_lon)
            current_lat, current_lon = current_location # Update for next iteration
            history.append(current_location)

            # Keep only a few recent points for visualization to avoid clutter
            display_history = history[-5:] # Display last 5 points

            print(f"Updating map with new location: Lat={current_lat:.4f}, Lon={current_lon:.4f} (Showing {len(display_history)} recent points)")
            create_and_display_map(display_history, current_location)
            time.sleep(UPDATE_INTERVAL_SECONDS)
        print("Live feed stopped.")

# --- Control Widgets ---
stop_flag = Bool(False, help="Set to True to stop the simulation").tag(sync=True)
start_button = widgets.Button(description="Start Live Feed")
stop_button = widgets.Button(description="Stop Live Feed")
output = widgets.Output()

def on_start_button_clicked(b):
    stop_flag.value = False
    with output:
        print("Starting live feed...")
    simulate_live_feed(stop_flag, output)

def on_stop_button_clicked(b):
    stop_flag.value = True
    with output:
        print("Stopping live feed...")

start_button.on_click(on_start_button_clicked)
stop_button.on_click(on_stop_button_clicked)

# Display the controls and the output area
display(widgets.HBox([start_button, stop_button]), output)

### How to Use:

1.  **Run all cells** above.
2.  Click the **'Start Live Feed'** button to begin the simulation.
3.  The map will update every 5 seconds with a new random location.
4.  Click the **'Stop Live Feed'** button to halt the updates.

This template can be extended by integrating with actual API calls to fetch data, rather than generating random locations. You can also customize the map's appearance and add more data visualization features.

## Streamlit Version of the Interactive Dashboard

This section provides a revised version of the dashboard code, adapted for deployment with [Streamlit](https://streamlit.io/).

**Key Changes:**

*   **Streamlit Widgets:** Replaces `ipywidgets` with `st.button` for controls.
*   **Session State:** Uses `st.session_state` to persist variables (like `current_lat`, `current_lon`, `history`, and `is_running`) across Streamlit reruns.
*   **`streamlit_folium`:** Integrates `folium` maps seamlessly into Streamlit.
*   **Continuous Updates:** The `time.sleep` and `st.rerun()` pattern is used to simulate the periodic updates, triggering the Streamlit app to re-execute and refresh the map.

In [ ]:
# Install necessary libraries for Streamlit
!pip install streamlit streamlit-folium

In [ ]:
import streamlit as st
import folium
from streamlit_folium import st_folium
import random
import time

# --- Configuration Parameters ---
INITIAL_LOCATION = [34.0522, -118.2437] # Los Angeles, for example
UPDATE_INTERVAL_SECONDS = 5

# --- Data Simulation Function ---
def generate_random_location(current_lat, current_lon, step_size=0.01):
    """Simulates a new location slightly perturbed from the current one."""
    new_lat = current_lat + random.uniform(-step_size, step_size)
    new_lon = current_lon + random.uniform(-step_size, step_size)
    return [new_lat, new_lon]

# --- Streamlit Application ---
st.title("Live Location Dashboard (Streamlit)")

# Initialize session state variables
if 'is_running' not in st.session_state:
    st.session_state.is_running = False
if 'current_lat' not in st.session_state:
    st.session_state.current_lat = INITIAL_LOCATION[0]
if 'current_lon' not in st.session_state:
    st.session_state.current_lon = INITIAL_LOCATION[1]
if 'history' not in st.session_state:
    st.session_state.history = []

# Buttons to start/stop the feed
col1, col2 = st.columns(2)

with col1:
    if st.button("Start Live Feed"):
        st.session_state.is_running = True
        st.session_state.current_lat = INITIAL_LOCATION[0] # Reset on start
        st.session_state.current_lon = INITIAL_LOCATION[1] # Reset on start
        st.session_state.history = []
        st.experimental_rerun()

with col2:
    if st.button("Stop Live Feed"):
        st.session_state.is_running = False
        st.experimental_rerun()

# Placeholder for the map and status messages
map_placeholder = st.empty()
status_placeholder = st.empty()

# Simulate live feed logic
if st.session_state.is_running:
    status_placeholder.info("Live feed is running...")

    # Generate new location
    new_lat, new_lon = generate_random_location(st.session_state.current_lat, st.session_state.current_lon)
    st.session_state.current_lat = new_lat
    st.session_state.current_lon = new_lon
    st.session_state.history.append([new_lat, new_lon])

    # Keep only a few recent points for visualization to avoid clutter
    display_history = st.session_state.history[-5:]

    with map_placeholder.container():
        status_placeholder.text(f"Updating map with new location: Lat={new_lat:.4f}, Lon={new_lon:.4f} (Showing {len(display_history)} recent points)")

        # Create the folium map
        m = folium.Map(location=[st.session_state.current_lat, st.session_state.current_lon], zoom_start=12)
        for lat, lon in display_history:
            folium.Marker([lat, lon]).add_to(m)

        # Display the map using streamlit_folium
        st_folium(m, width=700, height=500)

    # Wait for the interval and trigger a rerun
    time.sleep(UPDATE_INTERVAL_SECONDS)
    st.experimental_rerun()
else:
    status_placeholder.warning("Live feed is stopped. Click 'Start Live Feed' to begin.")



### To Run this Streamlit Application:

1.  **Save the code** in the last Python cell as a `.py` file (e.g., `app.py`).
2.  **Open your terminal or command prompt**.
3.  **Navigate to the directory** where you saved `app.py`.
4.  **Run the command:** `streamlit run app.py`

This will launch the Streamlit application in your web browser, providing the interactive dashboard. Note that `st.experimental_rerun()` is used here to force the application to re-execute and update the map, which is common for simulating real-time updates in Streamlit within a basic Python script. For more robust real-time applications, you might consider external data sources and callback mechanisms.